In [9]:
import graphviz

def create_quantum_diagram():
    # Initialize the directed graph
    dot = graphviz.Digraph('Quantum_Architecture', format='png')
    
    # Global graph attributes for orthogonal routing and spacing
    dot.attr(rankdir='TB', splines='ortho', nodesep='0.5', ranksep='0.6')
    dot.attr('node', shape='box', style='rounded,filled', fontname='Arial, sans-serif', margin='0.25,0.15', penwidth='1.5')
    dot.attr('edge', color='#b3b3b3', penwidth='1.5', arrowsize='0.8')

    # --- 1. Top Level Input ---
    dot.node('input', 
             '<<B>Quantum circuit input</B><BR/><FONT POINT-SIZE="11" COLOR="#555555"> </FONT>>', 
             fillcolor='#f6f4f0', color='#dcd8cf', fontcolor='#333333')

    # --- 2. Host CPU Orchestration Cluster ---
    with dot.subgraph(name='cluster_host') as c_host:
        c_host.attr(label='Host CPU orchestration', style='dashed', color='#c0c0c0', fontcolor='#555555', fontname='Arial', labeljust='l')
        
        # Row 1: Analyzers and Optimizers
        c_host.node('analyzer', '<<B>Circuit analyzer</B><BR/><FONT POINT-SIZE="11">Entanglement detection</FONT>>', 
                    fillcolor='#efebfc', color='#c4bced', fontcolor='#311b92')
        c_host.node('optimizer', '<<B>Path optimizer</B><BR/><FONT POINT-SIZE="11">Transfer-aware cost</FONT>>', 
                    fillcolor='#efebfc', color='#c4bced', fontcolor='#311b92')
        c_host.node('slicer', '<<B>WRAM slicer</B><BR/><FONT POINT-SIZE="11">64 KiB tile bounds</FONT>>', 
                    fillcolor='#efebfc', color='#c4bced', fontcolor='#311b92')
        
        # Row 2 & 3: Conversion and Routing
        c_host.node('conversion', '<<B>Data format conversion</B><BR/><FONT POINT-SIZE="11">BFP · int8 · fixed-point quantization</FONT>>', 
                    fillcolor='#fdf3ed', color='#dfa891', fontcolor='#6e2c00')
        c_host.node('router', '<<B>Dynamic heuristic router</B><BR/><FONT POINT-SIZE="11">State-based kernel dispatch</FONT>>', 
                    fillcolor='#fcf4e3', color='#dfc28e', fontcolor='#5d4037')

    # --- 3. UPMEM Execution Layer Cluster ---
    with dot.subgraph(name='cluster_upmem') as c_upmem:
        c_upmem.attr(label='UPMEM execution layer', style='dashed', color='#c0c0c0', fontcolor='#555555', fontname='Arial', labeljust='l')
        
        # Row 1: Execution Routes
        c_upmem.node('route_h', '<<B>Heuristic route</B><BR/><FONT POINT-SIZE="11">Gate merge / row swap</FONT>>', 
                     fillcolor='#e8f6f0', color='#9ecab6', fontcolor='#004d40')
        c_upmem.node('route_s', '<<B>Sparse route</B><BR/><FONT POINT-SIZE="11">SparseP SpMV kernels</FONT>>', 
                     fillcolor='#e8f6f0', color='#9ecab6', fontcolor='#004d40')
        c_upmem.node('route_d', '<<B>Dense GEMM route</B><BR/><FONT POINT-SIZE="11">ATiM tiled GEMM</FONT>>', 
                     fillcolor='#e8f6f0', color='#9ecab6', fontcolor='#004d40')
        
        # Row 2: TransPimLib
        c_upmem.node('transpim', '<<B>TransPimLib support</B><BR/><FONT POINT-SIZE="11">CORDIC · LUT transcendentals</FONT>>', 
                     fillcolor='#f1f8e9', color='#b2cc94', fontcolor='#33691e')

    # --- 4. Bottom Level Outputs ---
    dot.node('aggregation', '<<B>Host aggregation</B><BR/><FONT POINT-SIZE="11">PID-Comm-style reduction</FONT>>', 
             fillcolor='#ebf4fd', color='#a9c8eb', fontcolor='#0d47a1')
    dot.node('output', '<<B>Amplitude output</B>>', 
             fillcolor='#f6f4f0', color='#dcd8cf', fontcolor='#333333')

    # --- 5. Defining Edges (Connections) ---
    # Input to top row
    dot.edge('input', 'analyzer')
    dot.edge('input', 'optimizer')
    dot.edge('input', 'slicer')

    # Top row to conversion
    dot.edge('analyzer', 'conversion')
    dot.edge('optimizer', 'conversion')
    dot.edge('slicer', 'conversion')

    # Conversion to router
    dot.edge('conversion', 'router')

    # Router to execution routes (With floating text label on the middle edge)
    dot.edge('router', 'route_h')
    dot.edge('router', 'route_s', label='  64 KiB DMA  tiles', fontcolor='#555555', fontsize='11', labeljust='l')
    dot.edge('router', 'route_d')

    # Execution routes to TransPimLib (using dashed lines as in the diagram)
    dot.edge('route_h', 'transpim', style='dashed')
    dot.edge('route_s', 'transpim', style='dashed')
    dot.edge('route_d', 'transpim', style='dashed')

    # Bottom connections
    dot.edge('transpim', 'aggregation')
    dot.edge('aggregation', 'output')

    # Render the graph to a file
    dot.render(cleanup=True)
    print("Diagram successfully generated as 'Quantum_Architecture.png'")

if __name__ == '__main__':
    create_quantum_diagram()

Diagram successfully generated as 'Quantum_Architecture.png'
